In [0]:
from pyspark.sql import functions as F

# CONFIG
SILVER_STORE_TABLE = "mini_project_team_a3t7.silver.store_master"
GOLD_STORE_TABLE = "mini_project_team_a3t7.gold.dim_store"
CHECKPOINT_GOLD_SIMPLE = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/dim_store"

# READ STREAM
df_store_stream = (
    spark.readStream
    .option("skipChangeCommits", "true")   # ⭐ FIX
    .table(SILVER_STORE_TABLE)
)

# TRANSFORM
dim_store_final = (
    df_store_stream
    .withColumn(
        "years_in_operation",
        F.round(F.datediff(F.current_date(), F.col("opening_date")) / 365.25, 1)
    )

    .withColumn(
        "store_size_tier",
        F.when(F.col("store_type") == "Warehouse", "Large")
         .when(F.col("store_type") == "Superstore", "Medium")
         .otherwise("Small")
    )

    .withColumn("_gold_processed_at", F.current_timestamp())

    .select(
        "store_id",
        "store_name",
        "region",
        "city",
        "store_type",
        "store_size_tier",
        "opening_date",
        "years_in_operation",
        "_gold_processed_at"
    )
)

# WRITE STREAM
query = (
    dim_store_final.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_GOLD_SIMPLE)
    .trigger(availableNow=True)
    .toTable(GOLD_STORE_TABLE)
)

In [0]:
from pyspark.sql import functions as F

# 1. CONFIGURATION
SILVER_SUPPLIER_TABLE = "mini_project_team_a3t7.silver.supplier_master"
GOLD_SUPPLIER_TABLE = "mini_project_team_a3t7.gold.dim_supplier"
CHECKPOINT_GOLD_SUPPLIER = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/dim_supplier"

# 2. READ STREAM FROM SILVER
df_sup_stream = (
    spark.readStream
    .option("skipChangeCommits", "true")   # ⭐ FIX ADDED
    .table(SILVER_SUPPLIER_TABLE)
)

# 3. TRANSFORMATIONS
dim_supplier_final = (
    df_sup_stream

    # Risk Tier
    .withColumn(
        "baseline_risk_tier",
        F.when(F.col("on_time_delivery_pct") >= 90, "Low")
         .when(F.col("on_time_delivery_pct") >= 80, "Medium")
         .otherwise("High")
    )

    # Performance Band
    .withColumn(
        "performance_band",
        F.when(F.col("performance_rating") >= 4.5, "Excellent")
         .when(F.col("performance_rating") >= 3.5, "Good")
         .otherwise("Average")
    )

    # Gold Metadata
    .withColumn("_gold_processed_at", F.current_timestamp())

    # Final Columns
    .select(
        "supplier_id",
        "supplier_name",
        "category",
        "performance_rating",
        "on_time_delivery_pct",
        "baseline_risk_tier",
        "performance_band",
        "_gold_processed_at"
    )
)

# 4. WRITE STREAM
query = (
    dim_supplier_final.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_GOLD_SUPPLIER)
    .trigger(availableNow=True)
    .toTable(GOLD_SUPPLIER_TABLE)
)

### dim_product

In [0]:
from pyspark.sql.functions import col, when, round

# ── Step 1: Read from Silver ───────────────────────────────────────────
df_prod_current = spark.table("mini_project_team_a3t7.silver.product_master")



# ── Step 2: Create Gold Dimension ──────────────────────────────────────
dim_product = (
    df_prod_current

    # Profit margin for BI analytics
    .withColumn(
        "profit_margin_pct",
        round(
            (col("selling_price") - col("cost_price")) /
            col("selling_price") * 100,
            2
        )
    )

    # Price segmentation
    .withColumn(
        "price_tier",
        when(col("selling_price") >= 500, "Premium")
        .when(col("selling_price") >= 100, "Mid-range")
        .otherwise("Budget")
    )

    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "supplier_id",
        "cost_price",
        "selling_price",
        "profit_margin_pct",
        "price_tier",
        "weight",
        "dimensions",
        "status"
    )
)

# ── Step 3: Write to Gold ──────────────────────────────────────────────
(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mini_project_team_a3t7.gold.dim_product")
)
